# Chapter 4: Time-Series Data in Pandas

A beginner-friendly workbook packed with detailed explanations, visual examples, and practice exercises:
1. **4.1 Creating Time Series**: Parsing dates with `pd.to_datetime()`, custom `format` codes, combining date columns, and generating sequences with `pd.date_range()`
2. **4.2 Slicing, Timedelta Math & Resampling**: Partial date slicing with `.loc[]`, calculating elapsed time with `Timedelta`, and changing frequencies with `.resample()`

---
---

## 4.1 Creating Time Series (`to_datetime` & `date_range`)

### Concept Summary
**Time-series data** is simply data where every row is stamped with a specific date or time (such as hourly weather temperatures, daily stock prices, or website clicks per minute).

In Python and Pandas, real **datetime objects** store individual time pieces (`year`, `month`, `day`, `hour`, `minute`, `second`, `microsecond`, and optional `tzinfo` for time zones). Converting plain text dates into real datetime objects unlocks smart calendar math, automatic weekday detection, and easy date slicing.

---

### Tools for Creating Time Series: Deep Dive with Examples

#### 1. Converting Strings to Dates (`pd.to_datetime()`)
Pandas automatically recognizes most common date styles without any extra instructions:
- **Standard Dates:** `'2024-08-05'`, `'05/08/2024'`, `'08/05/2024'`
- **Date + Time:** `'2024-08-05 14:30:00'`
- **Month Names:** `'05-Aug-2024'`, `'August 5, 2024'`
- **No Separators:** `'20240805'` (read as `YYYYMMDD`)

```python
d = pd.to_datetime('2024-08-05')
print(d)         # Output: 2024-08-05 00:00:00
print(d.year)    # Output: 2024
print(d.month)   # Output: 8
```

#### 2. Handling Unusual Date Formats (`format=...`)
If your text contains weird symbols (like `@`) or unusual ordering, tell Pandas how to read it using the `format` parameter and format codes:

| Code | Meaning | Example |
| :---: | :--- | :--- |
| `%Y` | 4-digit year | `2024` |
| `%y` | 2-digit year | `24` |
| `%m` | 2-digit month (`01`–`12`) | `08` |
| `%B` | Full month name | `August` |
| `%b` | Short month name | `Aug` |
| `%d` | 2-digit day (`01`–`31`) | `05` |
| `%H` | Hour in 24-hour clock (`00`–`23`) | `14` |
| `%M` | Minute (`00`–`59`) | `30` |
| `%S` | Second (`00`–`59`) | `00` |

**Example (Parsing a custom string with `@`):**
```python
date_str = '2024 August 05 @ 14:30:00'
clean_date = pd.to_datetime(date_str, format='%Y %B %d @ %H:%M:%S')
print(clean_date)  # Output: 2024-08-05 14:30:00
```

#### 3. Combining Separate Year, Month, Day Columns
If a DataFrame stores `'year'`, `'month'`, and `'day'` in separate columns, pass all three into `pd.to_datetime()` to build a single date index:
```python
df = pd.DataFrame({
    'year':  [2024, 2025, 2026],
    'month': [8, 9, 10],
    'day':   [5, 15, 25],
    'vals':  [10, 20, 30]
})

# Combine into one datetime column, make it the index, and remove old columns
df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
df.set_index('date', inplace=True)
df.drop(columns=['year', 'month', 'day'], inplace=True)

# Output:
#             vals
# date            
# 2024-08-05    10
# 2025-09-15    20
# 2026-10-25    30
```

#### 4. Generating Time Sequences (`pd.date_range()`)
`pd.date_range(start, end, periods, freq)` builds an evenly spaced sequence of timestamps (`DatetimeIndex`) automatically:
- **Method A (`start` + `end` + `freq`):** Steps from start to end at a fixed interval (e.g., `'D'` = daily, `'2h'` = every 2 hours, `'ME'` = month end).
  ```python
  # Every 2 hours from July 21 to July 22
  dr = pd.date_range(start='2024-07-21', end='2024-07-22', freq='2h')
  ```
- **Method B (`start` + `periods`):** Generates a fixed count of timestamps starting from `start`.
  ```python
  # 5 daily dates starting Jan 1, 2024
  pd.date_range(start='2024-01-01', periods=5, freq='D')
  ```
- **Method C (`start` + `end` + `periods`):** Splits the exact span between `start` and `end` into `periods` equally spaced points.

In [ ]:
# --- Demo: Creating Time-Series DataFrames ---
import pandas as pd
import numpy as np

# 1. Parsing a custom date string using format codes
raw_stamp = '2024 August 05 @ 14:30:00'
parsed = pd.to_datetime(raw_stamp, format='%Y %B %d @ %H:%M:%S')
print("Parsed Timestamp:", parsed, "| Year:", parsed.year)

# 2. Combining separate columns into a DatetimeIndex
df_parts = pd.DataFrame({
    'year':  [2024, 2025, 2026],
    'month': [8, 9, 10],
    'day':   [5, 15, 25],
    'vals':  [1, 2, 3]
})
df_parts['date'] = pd.to_datetime(df_parts[['year', 'month', 'day']])
df_parts.set_index('date', inplace=True)
df_parts.drop(columns=['year', 'month', 'day'], inplace=True)

print("\n=== DataFrame Indexed by Date ===")
print(df_parts)

# 3. Generating a synthetic time series with pd.date_range
dr = pd.date_range(start='1999-07-21', end='1999-09-21', freq='2h')
ts_df = pd.DataFrame({'data': np.random.randn(len(dr))}, index=dr)

print("\n=== Generated 2-Hourly Time Series (First 5 Rows) ===")
print(ts_df.head())

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 1: Parsing Timestamps and Building a Time-Indexed Table

**Goal:** Convert messy date strings and multi-column date records into clean Pandas `DatetimeIndex` tables.

**Requirements:**
1. Parse the custom timestamp string `'15-Nov-2025 | 09:45'` into a Pandas datetime object using the proper `format` codes, and print its `month` and `minute` attributes.
2. Create a DataFrame named `sensor_logs` with columns:
   - `'year'`: `[2025, 2025, 2025]`
   - `'month'`: `[11, 11, 12]`
   - `'day'`: `[10, 20, 1]`
   - `'temp_c'`: `[18.4, 16.2, 14.9]`
3. Combine `'year'`, `'month'`, and `'day'` into a single datetime index, remove the three original date columns, and print the resulting DataFrame.
4. Using `pd.date_range()`, generate a sequence of `7` daily dates starting on `'2026-01-01'` and pair them as the index for a new DataFrame containing `7` random numbers.

<details>
<summary>💡 Need a hint?</summary>

- Match every separator (`-`, `|`, `:`) and character in `'15-Nov-2025 | 09:45'` when writing your format string.
- Remember `%b` matches 3-letter abbreviated month names like `'Nov'`.
- Passing a list of column names `df[['year', 'month', 'day']]` into `pd.to_datetime()` builds dates automatically.
- Use `periods=7` and `freq='D'` inside `pd.date_range()` for step 4.
</details>

In [ ]:
# --- Exercise 1: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 4.2 Slicing, Timedelta Math, and Resampling

### Concept Summary
Once a DataFrame has a `DatetimeIndex`, you can slice entire months with a short string, calculate exact durations between timestamps, and summarize data into weekly or monthly averages.

---

### Core Time-Series Operations

#### 1. Smart Date Slicing with `.loc[]`
Use `df.loc[]` with human-readable date strings. Pandas supports **partial string indexing**, meaning you can grab an entire year, month, or day just by typing that part of the date:
- **Slice an exact time range (inclusive):**
  ```python
  df.loc['1999-08-21 08:00:00':'1999-08-22 08:00:00']
  ```
- **Grab an entire month:**
  ```python
  df.loc['1999-09']  # Returns every row in September 1999!
  ```
- **Grab a single day:**
  ```python
  df.loc['1999-09-20']  # Returns all hourly rows on Sept 20, 1999
  ```

#### 2. Date Arithmetic & `Timedelta` Objects
Subtracting two datetime objects produces a **`Timedelta`** object that measures the exact elapsed time (down to the nanosecond), automatically accounting for month lengths and leap years.
- **Example (Calculating time elapsed since the start of the dataset):**
  ```python
  df['Time_from_start'] = df.index - df.index[0]
  # Produces values like: '61 days 04:00:00'
  ```

#### 3. Resampling (Changing Time Frequency)
**Resampling** works just like `.groupby()`, except it groups rows by **time buckets** instead of text categories.
- **Downsampling:** Converting high-frequency data into larger time buckets (e.g., hourly data $\rightarrow$ weekly sums or monthly averages).
- **Upsampling:** Converting low-frequency data into smaller time buckets (e.g., monthly data $\rightarrow$ daily rows).

```text
=========================================================================================
                         DOWNSAMPLING WORKFLOW (df.resample)
=========================================================================================

  [ 2-Hourly Readings ]             [ Time Buckets ]              [ Summary Table ]
  1999-07-21 00:00 -> 0.5
  1999-07-21 02:00 -> 1.5   --->   All of July 1999   --mean()--> 1999-07-31 -> 0.008
  ...                              All of Aug  1999   --mean()--> 1999-08-31 -> 0.048
  1999-09-21 00:00 -> 0.2          All of Sept 1999   --mean()--> 1999-09-30 -> -0.089
=========================================================================================
```

- **Syntax:** `df.resample(rule)['Column'].agg_func()`
- **Common Rules:** `'D'` (Daily), `'W'` (Weekly), `'ME'` (Month End — or `'M'` in older Pandas), `'YE'` (Year End).
- **Example:**
  ```python
  monthly_mean = df.resample('ME')['data'].mean()  # Monthly average
  weekly_sum   = df.resample('W')['data'].sum()    # Weekly total
  ```

In [ ]:
# --- Demo: Date Slicing, Timedelta & Resampling ---
import pandas as pd
import numpy as np

# Setup 2-hourly synthetic dataset
np.random.seed(42)
dr = pd.date_range(start='1999-07-21', end='1999-09-21', freq='2h')
df = pd.DataFrame({'data': np.random.randn(len(dr))}, index=dr)

# 1. Slice a specific time window
dayslice = df.loc['1999-08-21 08:00:00':'1999-08-22 08:00:00']
print("=== Slice from Aug 21 08:00 to Aug 22 08:00 ===")
print(dayslice.head())

# 2. Partial string slicing (grab all of September)
sept_data = df.loc['1999-09']
print("\n=== Total Rows in September 1999:", len(sept_data))

# 3. Timedelta arithmetic (elapsed time from first timestamp)
df['Time_from_start'] = df.index - df.index[0]
print("\n=== Sample Rows on 1999-09-20 with Timedelta ===")
print(df.loc['1999-09-20'].head())

# 4. Resampling (Monthly mean and Weekly sum)
print("\n=== Monthly Mean (Downsampling) ===")
print(df.resample('ME')['data'].mean())

print("\n=== Weekly Sum (First 5 Weeks) ===")
print(df.resample('W')['data'].sum().head())

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 2: Time-Window Filtering, Elapsed Duration & Resampling

**Goal:** Slice hourly energy usage records by date labels, calculate elapsed time using `Timedelta`, and aggregate data into daily and weekly totals.

**Requirements:**
1. Create a `DatetimeIndex` named `hours` running every 1 hour (`'1h'`) from `'2026-03-01'` to `'2026-03-21'`.
2. Create a DataFrame `energy_df` indexed by `hours` with a single column `'kwh'` populated with random floats between `1.0` and `5.0` (using `np.random.uniform(1.0, 5.0, len(hours))`).
3. Using partial string indexing with `.loc[]`, extract and print all hourly readings recorded on `'2026-03-10'`.
4. Add a new column `'Elapsed'` that stores the `Timedelta` difference between each row's timestamp and the very first timestamp in the table.
5. Downsample `energy_df` to compute and display:
   - The **daily average** `'kwh'` usage (first 5 days).
   - The **weekly total sum** of `'kwh'` usage.

<details>
<summary>💡 Need a hint?</summary>

- For step 3, passing `'2026-03-10'` directly into `df.loc[...]` selects all 24 hours of that day.
- For step 4, subtract `df.index[0]` from `df.index`.
- For step 5, use `.resample('D')` for daily buckets and `.resample('W')` for weekly buckets, followed by `['kwh'].mean()` or `['kwh'].sum()`.
</details>

In [ ]:
# --- Exercise 2: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>